<a href="https://www.kaggle.com/code/nicapotato/titanic-feature-engineering?scriptVersionId=320617598" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Another Titanic Feature Engineering Notebook!
### With Nick Brooks

## Includes:
- Filling missing values
- Parsing the passenger title
- Map categorical strings to number
- Standardizing
- Dummy Variables (Optional)


In [1]:
# General
import numpy as np
import pandas as pd

# Normalizer
from sklearn import preprocessing

In [2]:
# Read Data
train_df = pd.read_csv("../input/train.csv", index_col='PassengerId')
test_df = pd.read_csv("../input/test.csv", index_col='PassengerId')
Survived = train_df['Survived'].copy()
train_df = train_df.drop('Survived', axis=1)

In [3]:
test_df.shape, train_df.shape

((418, 10), (891, 10))

In [4]:
# Combine Test and Train to perform feature engineering all at once
df = pd.concat([test_df, train_df])
traindex = train_df.index
testdex = test_df.index
print(test_df.equals(df.loc[testdex,:]))
print(train_df.equals(df.loc[traindex,:]))
del train_df
del test_df

True
True


### Before

In [5]:
df.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,
892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


### Missing Values

In [6]:
# Proportion Missing Table:
settypes=df.dtypes.reset_index()
def missing(df):
    missing = df.isnull().sum(axis=0).reset_index()
    missing.columns = ['column_name', 'missing_count']
    missing['missing_ratio'] = missing['missing_count'] / df.shape[0]
    missing = pd.merge(missing,settypes, left_on='column_name', right_on='index',how='inner')
    missing = missing.loc[(missing['missing_ratio']>0)]\
    .sort_values(by=["missing_ratio"], ascending=False)
    return missing

In [7]:
mis = missing(df)
mis

,column_name,missing_count,missing_ratio,index,0
8,Cabin,1014,0.774637,Cabin,object
3,Age,263,0.200917,Age,float64
9,Embarked,2,0.001528,Embarked,object
7,Fare,1,0.000764,Fare,float64


# New Features

In [8]:
# New Variables engineering, heavily influenced by:
# Kaggle Source- https://www.kaggle.com/arthurtok/introduction-to-ensembling-stacking-in-python
# Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
# Name Length
df['Name_length'] = df['Name'].apply(len)
# Is Alone?
df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

## Title

In [9]:
# Title: (Source)
# Kaggle Source- https://www.kaggle.com/ash316/eda-to-prediction-dietanic
df['Title']=0
df['Title']=df.Name.str.extract('([A-Za-z]+)\.') #lets extract the Salutations
df['Title'].replace(['Mlle','Mme','Ms','Dr','Major','Lady','Countess','Jonkheer','Col',
                         'Rev','Capt','Sir','Don'],['Miss','Miss','Miss','Mr','Mr','Mrs','Mrs','Other','Other','Other','Mr','Mr','Mr'],inplace=True)

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:4: FutureWarning: currently extract(expand=None) means expand=False (return Index/Series/DataFrame) but in a future version of pandas this will be changed to expand=True (return DataFrame)
  after removing the cwd from sys.path.


## Age 

In [10]:
df.loc[(df.Age.isnull())&(df.Title=='Mr'),'Age']= df.Age[df.Title=="Mr"].mean()
df.loc[(df.Age.isnull())&(df.Title=='Mrs'),'Age']= df.Age[df.Title=="Mrs"].mean()
df.loc[(df.Age.isnull())&(df.Title=='Master'),'Age']= df.Age[df.Title=="Master"].mean()
df.loc[(df.Age.isnull())&(df.Title=='Miss'),'Age']= df.Age[df.Title=="Miss"].mean()
df.loc[(df.Age.isnull())&(df.Title=='Other'),'Age']= df.Age[df.Title=="Other"].mean()
df = df.drop('Name', axis=1)

## Fill NA

In [11]:
# Fill NA
# Categoricals Variable
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode().iloc[0])
# Continuous Variable
df['Fare'] = df['Fare'].fillna(df['Fare'].mean())

## String to Numeric

In [12]:
## Assign Binary to Sex str
df['Sex'] = df['Sex'].map( {'female': 1, 'male': 0} ).astype(int)
# Title
#df['Title'] = df['Title'].map( {'Mr': 0, 'Mrs': 1, 'Miss': 2, 'Master':3, 'Rare':4} ).astype(int)
# Embarked
df['Embarked'] = df['Embarked'].map( {'Q': 0, 'S': 1, 'C': 2} ).astype(int)

# Get Rid of Ticket and Cabin Variable
df= df.drop(['Ticket', 'Cabin'], axis=1)

## Standardization

In [13]:
# Scaling between -1 and 1. Good practice for continuous variables.
from sklearn import preprocessing
for col in ['Fare','Age','Name_length']:
    transf = df[col].reshape(-1,1)
    scaler = preprocessing.StandardScaler().fit(transf)
    df[col] = scaler.transform(transf)

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:4: FutureWarning: reshape is deprecated and will raise in a subsequent release. Please use .values.reshape(...) instead
  after removing the cwd from sys.path.
/opt/conda/lib/python3.6/site-packages/sklearn/utils/validation.py:475: DataConversionWarning: Data with input dtype int64 was converted to float64 by StandardScaler.
  warnings.warn(msg, DataConversionWarning)


#### Recombine

In [14]:
train_df = df.loc[traindex, :]
train_df['Survived'] = Survived

#### Create File

In [15]:
train_df.to_csv('clean_train_nick.csv',header=True,index=True)
df.loc[testdex, :].to_csv('clean_test_nick.csv',header=True,index=True)

#### View Output

In [16]:
train_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Name_length,IsAlone,Title,Survived
PassengerId,,,,,,,,,,,,
1,3,0,-0.600854,1,0,-0.503595,1,2,-0.434672,0,Mr,0
2,1,1,0.612019,1,0,0.734503,2,2,2.511806,0,Mrs,1
3,3,1,-0.297635,0,0,-0.490544,1,1,-0.539904,1,Miss,1
4,1,1,0.384606,1,0,0.382925,1,2,1.775186,0,Mrs,1
5,3,0,0.384606,0,0,-0.488127,1,1,-0.329441,1,Mr,0


In [17]:
df.describe()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Name_length,IsAlone
count,1309.000000,1309.000000,1.309000e+03,1309.000000,1309.000000,1.309000e+03,1309.000000,1309.000000,1.309000e+03,1309.000000
mean,2.294882,0.355997,1.142028e-16,0.498854,0.385027,7.561221e-17,1.112299,1.883881,-1.435911e-16,0.603514
std,0.837836,0.478997,1.000382e+00,1.041658,0.865560,1.000382e+00,0.536505,1.583639,1.000382e+00,0.489354
min,1.000000,0.000000,-2.255667e+00,0.000000,0.000000,-6.437751e-01,0.000000,1.000000,-1.592217e+00,0.000000
25%,2.000000,0.000000,-6.133967e-01,0.000000,0.000000,-4.911082e-01,1.000000,1.000000,-7.503663e-01,0.000000
50%,3.000000,0.000000,5.582946e-03,0.000000,0.000000,-3.643001e-01,1.000000,1.000000,-2.242095e-01,1.000000
75%,3.000000,1.000000,4.604103e-01,1.000000,0.000000,-3.906640e-02,1.000000,2.000000,3.019473e-01,1.000000
max,3.000000,1.000000,3.795811e+00,8.000000,9.000000,9.262219e+00,2.000000,11.000000,5.773978e+00,1.000000


Perhaps it is unwise to perform standardization on the combined train and test set since it could give information away about the test set?

Suggestions and Improvement are welcomed!